# Fast spot quantification — per-bit foci intensity QC

Finds a bit with weak hybridization early, while it can still be redone. For
a few FOVs (`SELECTED_FOVS`) and every (round, color), measures each focus's
background-subtracted peak intensity:

1. Read `N_Z_SAMPLES` z-planes spread over the color's z-range.
2. Crop to the central `CROP_SIZE` square (avoids edge vignetting).
3. Max-project and detect foci (`detect_beads_2d`).
4. Peak value minus the crop's background median.

Cached in `SAMPLE_DIR/analysis/fast_spot_quantification/`, one CSV per
(round, color, FOV); re-runs compute only new ones (`OVERWRITE_OUTPUT = True`
redoes all).

The final plot has one row per color and one violin per round; a weak round
stands out as lower or tighter. A relative check, tuned by eye
(`FOCI_MIN_DIST_PX`, `FOCI_THRESH_SIGMA`, section 5's overlay).

## 1 — Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

MERCI_DIR  = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/during_imaging/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.config          import ExperimentConfig
from MERci.common.metadata        import ExperimentMetadata
from MERci.common.experiment_info import resolve_sample_identity, positions_file_tag
from MERci.common.io              import read_image_frames
from MERci.progress_display       import ProgressReporter
from MERci.plots.experiment_plots import get_merci_figures_dir
from MERci.analysis.fast_spot_quantification import (
    resolve_round_color_frame_indices, detect_foci_in_crop, spot_cache_path,
    compute_fov_round_color_spots, load_all_spots,
)

print(f"SAMPLE_DIR: {SAMPLE_DIR}")

## 2 — Parameters

In [ ]:
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
POSITIONS_TAG = positions_file_tag(SAMPLE_NAME, IMAGING_DIR)
IMAGE_SUFFIX  = ".zarr"   # same as HAL writes
NOTEBOOK_NAME = "fast_spot_quantification"   # figures folder name
MICROSCOPE    = "MF3"

# A few global fov_ids to sample.
SELECTED_FOVS = [0, 1, 2]

# Central crop side (px), away from edge vignetting, so no flat-field is needed.
CROP_SIZE = 400

# z-planes per (round, color), spread over the color's z-range and max-projected.
# 1 is fastest but misses out-of-focus foci; more reads more data.
N_Z_SAMPLES = 5

# Not bit colors: 405 (DAPI) and 488 (beads). Bit colors come from the frame tables.
EXCLUDED_COLORS = [405.0, 488.0]

# detect_beads_2d, tuned by eye on section 5's overlay: minimum foci spacing (px)
# and threshold above background (in background std).
FOCI_MIN_DIST_PX  = 4
FOCI_THRESH_SIGMA = 3.0

# False = skip (round, color, FOV) already cached; True = recompute all.
OVERWRITE_OUTPUT = False

# Plot font sizes.
PLOT_TITLE_FONTSIZE  = 14
PLOT_LABEL_FONTSIZE  = 12
PLOT_TICK_FONTSIZE   = 10
PLOT_LEGEND_FONTSIZE = 10

config = ExperimentConfig.from_sample_dir(
    SAMPLE_DIR,
    positions_txt  = SAMPLE_DIR / "positions" / f"positions_{POSITIONS_TAG}.txt",
    image_suffix   = IMAGE_SUFFIX,
    microscope     = MICROSCOPE,
)
meta = ExperimentMetadata.load(config.round_info_csv, config.positions_txt,
                                config.data_dir, image_suffix=config.image_suffix)

# Results (kept, like analysis/thumbnails/), not a recompute cache.
OUTPUT_DIR = config.analysis_dir / "fast_spot_quantification"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = get_merci_figures_dir(SAMPLE_DIR, "during_imaging", NOTEBOOK_NAME)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample name    : {SAMPLE_NAME}")
print(f"FOVs (total)   : {meta.n_fovs}")
print(f"Rounds         : {sorted(meta.rounds)}")
print(f"SELECTED_FOVS  : {SELECTED_FOVS}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")

## 3 — Resolve each round's real bit colors -> sampled z-frame indices

Per color, `N_Z_SAMPLES` frame indices spread over its z-range
(`get_all_color_frame_indices`), max-projected before detection.

In [ ]:
ROUND_COLOR_FRAME_INDICES = {}
for round_id in sorted(meta.rounds):
    cf = resolve_round_color_frame_indices(round_id, config, meta, EXCLUDED_COLORS, N_Z_SAMPLES)
    if cf:
        ROUND_COLOR_FRAME_INDICES[round_id] = cf
        n_samples_per_color = {c: len(f) for c, f in cf.items()}
        print(f"Round {round_id}: colors {sorted(cf)} -> n_z_samples per color: {n_samples_per_color}")
    else:
        print(f"Round {round_id}: no bit colors resolved")

## 4 — Foci detection

`detect_foci_in_crop` (`MERci.analysis.fast_spot_quantification`): crop,
max-project, background median, `detect_beads_2d`. Used by sections 5 and 6.

## 5 — Diagnostic: visualize detected foci on one example crop

Check `FOCI_MIN_DIST_PX`/`FOCI_THRESH_SIGMA` on the first imaged (round,
color, FOV) before the full run; re-run after changing them.

In [ ]:
EXAMPLE = None
for round_id, color_frames in ROUND_COLOR_FRAME_INDICES.items():
    series = meta.series_for_round(round_id)
    for color_nm, frame_indices in color_frames.items():
        for fov_id in SELECTED_FOVS:
            existing = [p for p in (s.resolve_path(fov_id, config.image_suffix) for s in series) if p.exists()]
            if existing:
                EXAMPLE = (round_id, color_nm, fov_id, frame_indices, existing[0])
                break
        if EXAMPLE:
            break
    if EXAMPLE:
        break

if EXAMPLE is None:
    print("No imaged (round, color, FOV) combination found yet for the diagnostic overlay -- "
          "run again once imaging has started.")
else:
    example_round_id, example_color_nm, example_fov_id, example_frame_indices, example_path = EXAMPLE
    frames = read_image_frames(example_path, example_frame_indices,
                                frame_width=config.frame_width, frame_height=config.frame_height)
    max_proj, bg_med, candidates = detect_foci_in_crop(frames, CROP_SIZE, FOCI_MIN_DIST_PX, FOCI_THRESH_SIGMA)

    vmin, vmax = np.percentile(max_proj, [1.0, 99.0])
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(max_proj, cmap="gray", vmin=vmin, vmax=vmax)
    if len(candidates):
        ax.scatter(candidates[:, 1], candidates[:, 0], s=40, facecolors="none",
                   edgecolors="red", linewidths=1.0)
    ax.set_title(f"round {example_round_id}, {example_color_nm:.0f} nm, FOV {example_fov_id} "
                 f"-- {len(candidates)} foci detected", fontsize=PLOT_TITLE_FONTSIZE)
    ax.axis("off")
    fig.tight_layout()
    fig_path = FIGURES_DIR / f"{NOTEBOOK_NAME}.example_foci_overlay.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved: {fig_path}")

## 6 — Run the quantification

Skips combinations already cached (unless `OVERWRITE_OUTPUT`). FOVs not imaged
yet are not cached, so a later run picks them up.

In [ ]:
combos = [
    (fov_id, round_id, color_nm, frame_indices)
    for round_id, color_frames in ROUND_COLOR_FRAME_INDICES.items()
    for color_nm, frame_indices in color_frames.items()
    for fov_id in SELECTED_FOVS
]

n_computed, n_skipped_cached, n_not_imaged = 0, 0, 0
reporter = ProgressReporter(total=len(combos), label="Quantifying foci")
for fov_id, round_id, color_nm, frame_indices in reporter.wrap(combos):
    cache_path = spot_cache_path(OUTPUT_DIR, round_id, color_nm, fov_id)
    if cache_path.exists() and not OVERWRITE_OUTPUT:
        n_skipped_cached += 1
        continue

    series = meta.series_for_round(round_id)
    df = compute_fov_round_color_spots(fov_id, round_id, color_nm, frame_indices, series,
                                        config, CROP_SIZE, FOCI_MIN_DIST_PX, FOCI_THRESH_SIGMA)
    if df is None:
        n_not_imaged += 1
        continue

    df.to_csv(cache_path, index=False)
    n_computed += 1

print(f"Computed: {n_computed} | Skipped (already cached): {n_skipped_cached} | Not yet imaged: {n_not_imaged}")

## 7 — Load cached results

In [ ]:
SPOTS_DF = load_all_spots(OUTPUT_DIR, ROUND_COLOR_FRAME_INDICES, SELECTED_FOVS)
n_combos = SPOTS_DF[["fov", "round", "color_nm"]].drop_duplicates().shape[0] if not SPOTS_DF.empty else 0
print(f"Loaded {len(SPOTS_DF)} foci across {n_combos} (fov, round, color) combination(s).")

## 8 — Plot: spot-intensity violin per bit color, across rounds

One row per color, one violin per round (x = `imaging_round`), all
`SELECTED_FOVS` pooled. A weak round looks lower or tighter.

In [ ]:
if SPOTS_DF.empty:
    print("No foci data yet -- nothing to plot. Run Section 6 once at least one combination is imaged.")
else:
    colors_present = sorted(SPOTS_DF["color_nm"].unique())
    fig, axes = plt.subplots(len(colors_present), 1, figsize=(10, 4 * len(colors_present)), squeeze=False)

    for ax, color_nm in zip(axes[:, 0], colors_present):
        sub = SPOTS_DF[SPOTS_DF["color_nm"] == color_nm]
        rounds_present = sorted(sub["round"].unique())
        per_round_data = [sub.loc[sub["round"] == r, "intensity"].values for r in rounds_present]
        # Skip rounds with no foci (violinplot fails on empty data).
        valid = [(r, d) for r, d in zip(rounds_present, per_round_data) if len(d) > 0]
        if valid:
            vp_rounds, vp_data = zip(*valid)
            parts = ax.violinplot(vp_data, positions=vp_rounds, showmedians=True)
            for body in parts["bodies"]:
                body.set_alpha(0.5)
            ax.set_xticks(vp_rounds)
        ax.set_title(f"{color_nm:.0f} nm", fontsize=PLOT_TITLE_FONTSIZE)
        ax.set_xlabel("Round", fontsize=PLOT_LABEL_FONTSIZE)
        ax.set_ylabel("Spot intensity (bg-subtracted)", fontsize=PLOT_LABEL_FONTSIZE)
        ax.tick_params(labelsize=PLOT_TICK_FONTSIZE)

    fig.tight_layout()
    fig_path = FIGURES_DIR / f"{NOTEBOOK_NAME}.spot_intensity_violin.png"
    fig.savefig(fig_path, dpi=150)
    plt.show()
    print(f"Saved: {fig_path}")